In [ ]:
%pip install openai --upgrade

## Personas of Thought
Asking the LLM to generate a crowd of personas to answer a question before aggregating their feedback.

This prompt inspired the new startup I started, [AskRally](https://askrally.com?utm_source=udemy), a virtual audience simulator for synthetic research.

In [18]:
import getpass
import os
from openai import OpenAI

# Set up your OpenAI client
if not os.getenv("OPENAI_API_KEY"):
    secret_key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = secret_key

In [19]:
client = OpenAI()
MODEL = 'gpt-5-mini'

In [20]:
question = "I'm starting a company that makes shoes that fit any foot size. What do you think of the name 'OmniFit'?"
number = 10

def call_openai(messages):
    client = OpenAI()
    return client.responses.create(
        model=MODEL,
        input=messages,
        text={"format": {"type": "text"}, "verbosity": "low"},
        reasoning={"effort": "minimal", "summary": "auto"},
    ).output_text

system_prompt = "You are a helpful and terse assistant."
question_prompt = "I want a paragraph response to the following question: {question}"

naive_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question_prompt.format(question=question)}
]

naive_response = call_openai(naive_messages)

print(naive_response)

"OmniFit" is a strong, concise name that conveys inclusivity and adaptability—good qualities for a shoe brand promising to fit any foot size. It's easy to pronounce and remember, and the "Omni-" prefix suggests comprehensive coverage while "Fit" communicates function. Check for trademark availability and domain names (omni fit/omnifit variations) and consider whether you want a more distinctive spelling if the market has similar names; otherwise, it's a solid, marketable choice.


In [28]:
experts_prompt = """
Don't ask for clarification, just do the task.
I want a paragraph response to the following question: {question}

First, name {number} world-class experts (past or present) who would be great at answering this?

Then for each expert, please answer the question critically from their perspective given their background and experience.

Finally, combine the responses into a single response as if these experts had collaborated in writing a joint anonymous answer.

## Expert names:
Expert name (relevant experience)
Expert name (relevant experience)
Expert name (relevant experience)
...

### Expert responses:
Expert name: <response>
Expert name: <response>
Expert name: <response>
...

### Final response:
<joint response>
"""

experts_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": experts_prompt.format(question=question, number=number)}
]

experts_response = call_openai(experts_messages)

print(experts_response)

Expert names:
Tim Brown (IDEO, design thinking and product design)
Don Norman (cognitive science, user-centered design)
Marty Neumeier (brand strategist, author of The Brand Gap)
Malcolm Gladwell (writer on social trends and branding)
Patricia Moore (industrial designer, ergonomic footwear expert)
Stuart Weitzman (fashion shoe designer)
Terry Moore (biomechanics and orthotics researcher)
Esther Dyson (investor, tech and consumer markets)
Philip Kotler (marketing professor, branding and positioning)
Jessica Alba (entrepreneur, Honest Company founder with consumer product experience)

Expert responses:
Tim Brown: OmniFit is clear and evocative—suggests inclusive design and adaptability. Strengthen it by ensuring the product experience visibly embodies that promise; otherwise name risks being hollow marketing.

Don Norman: The name communicates intent but says nothing about usability or comfort. Focus on demonstrating how OmniFit solves real user problems; consider a tagline clarifying be

In [29]:
personas_prompt = """
Don't ask for clarification, just do the task.

I want a paragraph response to the following question: {question}

First, name {number} demographic personas who would be relevant for answering this?

Then for each persona, please answer the question critically from their perspective given their background and experience.

Finally, combine the responses into a single response as if these personas had collaborated in writing a joint anonymous answer.

### Persona names:
Persona name (relevant demographics)
Persona name (relevant demographics)
Persona name (relevant demographics)
...

### Persona responses:
Persona name: <response>
Persona name: <response>
Persona name: <response>
...

### Final response:
<joint response>
"""

personas_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": personas_prompt.format(question=question, number=number)}
]

personas_response = call_openai(personas_messages)

print(personas_response)

Persona names:
Founder (early-stage entrepreneur, footwear background)
Product Designer (industrial designer, ergonomic focus)
Brand Strategist (marketing, naming expert)
Retail Buyer (footwear buyer for department stores)
End User — Athlete (runner, performance shoe user)
End User — Casual (everyday comfort-seeking shopper)
Podiatrist (medical foot specialist)
Manufacturing Manager (shoe production and supply chain)
Investor (VC focused on consumer goods)
Legal Counsel (trademark and IP attorney)

Persona responses:
Founder: OmniFit sounds strong and directly communicates the core promise—universal fit—but be wary of overpromising; ensure product delivers on “omni” or risk credibility loss.
Product Designer: The name aligns with ergonomic goals; it suggests adaptability and inclusivity, but it should pair with clear messaging about how fit is achieved to avoid vagueness.
Brand Strategist: OmniFit is memorable and descriptive, good for SEO; however, it’s slightly generic—consider a dis

In [30]:
def extract_final_response(response):
    return response.lower().split('final response:')[1].strip()

experts_final_response = extract_final_response(experts_response)
personas_final_response = extract_final_response(personas_response)

print("Naive: ", naive_response)
print("-"*100)
print("Experts: ", experts_final_response)
print("-"*100)
print("Personas: ", personas_final_response)


Naive:  "OmniFit" is a strong, concise name that conveys inclusivity and adaptability—good qualities for a shoe brand promising to fit any foot size. It's easy to pronounce and remember, and the "Omni-" prefix suggests comprehensive coverage while "Fit" communicates function. Check for trademark availability and domain names (omni fit/omnifit variations) and consider whether you want a more distinctive spelling if the market has similar names; otherwise, it's a solid, marketable choice.
----------------------------------------------------------------------------------------------------
Experts:  omnifit is a strong, concise name that effectively conveys universality and adaptability—good for attracting attention and telling a simple brand story. however, it also makes a bold promise: to avoid skepticism, pair the name with demonstrable user-centered design, clear communication about what “fits any foot” practically means, robust ergonomic and biomechanical validation, and compelling ae

In [31]:
judge_prompt = """I want you to compare two responses to the following question: {question}

Here are the two responses:

Response 1:
{response1}

Response 2:
{response2}

Please analyze both responses and choose which one is better. Consider factors like:
- Uniqueness of perspectives
- Nonobvious insights
- Sounds human not like an AI

## Your analysis:
<analysis>

## Your choice:
The better response is: Response <1 or 2>
"""

def judge_responses(response1, response2):
    judge_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": judge_prompt.format(
            question="How can I be more productive?",
            response1=response1,
            response2=response2
        )}
    ]

    judge_response = call_openai(judge_messages)

    choice_text = judge_response.split("The better response is: Response ")[1].strip()
    choice = int(choice_text[0])
    return {"choice": choice, "reason": judge_response}


naive_vs_experts = judge_responses(naive_response, experts_final_response)
print("# Naive vs experts:", naive_vs_experts["choice"])

naive_vs_personas = judge_responses(naive_response, personas_final_response)
print("# Naive vs personas:", naive_vs_personas["choice"])

experts_vs_personas = judge_responses(experts_final_response, personas_final_response)
print("# Experts vs personas:", experts_vs_personas["choice"])

print("-"*100)
print(naive_vs_experts)
print("-"*100)
print(naive_vs_personas)
print("-"*100)
print(experts_vs_personas)

# Naive vs experts: 2
# Naive vs personas: 2
# Experts vs personas: 1
----------------------------------------------------------------------------------------------------
{'choice': 2, 'reason': '<analysis>\nBoth responses evaluate the same name ("OmniFit"/"omnifit") rather than answering the user\'s stated question ("How can I be more productive?")—so neither addresses the original question. Focusing on the provided comparison:\n\nUniqueness of perspectives\n- Response 1: Brief, high-level branding take (name meaning, memorability, trademark/domain check). Conventional.\n- Response 2: Broader, more nuanced view linking name claims to product development, validation, marketing, and operations. Offers concrete actions to avoid overpromising.\n\nNonobvious insights\n- Response 1: Mostly obvious brand-name checklist; no deeper or surprising points.\n- Response 2: Raises nonobvious, practical risks (skepticism) and ties legal/technical/marketing remedies together (ergonomic/biomechanical v

### Bringing it all together

In [32]:
questions = [
    "What is the best way to learn a new skill?",
    "What is the best way to stay healthy?",
    "How can I be more productive?",
    "What will AI look like in 10 years?",
    "How do we end world hunger?",
]

number = 10

def run_test(question):

    naive_messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question_prompt.format(question=question)}
    ]

    naive_response = call_openai(naive_messages)

    print(naive_response)


    experts_messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": experts_prompt.format(question=question, number=number)}
    ]

    experts_response = call_openai(experts_messages)

    print(experts_response)


    personas_messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": personas_prompt.format(question=question, number=number)}
    ]

    personas_response = call_openai(personas_messages)

    print(personas_response)


    experts_final_response = extract_final_response(experts_response)
    personas_final_response = extract_final_response(personas_response)

    print("Naive: ", naive_response)
    print("-"*100)
    print("Experts: ", experts_final_response)
    print("-"*100)
    print("Personas: ", personas_final_response)

    naive_vs_experts = judge_responses(naive_response, experts_final_response)
    print("# Naive vs experts:", naive_vs_experts["choice"])

    naive_vs_personas = judge_responses(naive_response, personas_final_response)
    print("# Naive vs personas:", naive_vs_personas["choice"])

    experts_vs_personas = judge_responses(experts_final_response, personas_final_response)
    print("# Experts vs personas:", experts_vs_personas["choice"])

    print("-"*100)
    print(naive_vs_experts)
    print("-"*100)
    print(naive_vs_personas)
    print("-"*100)
    print(experts_vs_personas)

    return {
        "naive": naive_response,
        "experts": experts_final_response,
        "personas": personas_final_response,
        "naive_vs_experts": naive_vs_experts,
        "naive_vs_personas": naive_vs_personas,
        "experts_vs_personas": experts_vs_personas,
    }

results = {}

for question in questions:
    results[question] = run_test(question)

The best way to learn a new skill is to combine focused, deliberate practice with consistent feedback and spaced repetition: start by breaking the skill into small, specific subskills, set clear achievable goals, practice deliberately on the hardest or most important elements while avoiding mindless repetition, get immediate feedback from a coach, mentor, peer, or recording to correct errors, and schedule regular short sessions over weeks to reinforce retention. Supplement practice with targeted study—models, worked examples, and explanations—then gradually increase difficulty and apply the skill in real-world projects to build fluency and confidence. Stay patient, track progress, and iterate on your approach based on results.
Expert names:
1. Anders Ericsson (psychologist, deliberate practice researcher)
2. Carol Dweck (psychologist, mindset researcher)
3. Malcolm Gladwell (author, popularizer of expertise concepts)
4. Josh Waitzkin (chess prodigy, martial arts champion, author on lea

In [34]:
# Count wins for each approach
wins = {"naive": 0, "experts": 0, "personas": 0}
trials = {"naive": 0, "experts": 0, "personas": 0}

for question in results:
    # Naive vs experts
    if results[question]["naive_vs_experts"]["choice"] == 1:
        wins["naive"] += 1
    else:
        wins["experts"] += 1
    trials["naive"] += 1
    trials["experts"] += 1

    # Naive vs personas
    if results[question]["naive_vs_personas"]["choice"] == 1:
        wins["naive"] += 1
    else:
        wins["personas"] += 1
    trials["naive"] += 1
    trials["personas"] += 1

    # Experts vs personas
    if results[question]["experts_vs_personas"]["choice"] == 1:
        wins["experts"] += 1
    else:
        wins["personas"] += 1
    trials["experts"] += 1
    trials["personas"] += 1

# Calculate percentages
percentages = {
    approach: (wins[approach] / trials[approach]) * 100
    for approach in wins
}

print("\nWin percentages:")
for approach, percentage in percentages.items():
    print(f"{approach}: {percentage:.1f}%")



Win percentages:
naive: 0.0%
experts: 50.0%
personas: 100.0%
